# 08 - Kafka streaming to Iceberg - Solution


In [ ]:
from pyspark.sql import SparkSession, functions as F, types as T

spark = SparkSession.builder.appName("lesson08_streaming").getOrCreate()
schema = T.StructType([
    T.StructField("event_id", T.StringType()),
    T.StructField("event_ts", T.StringType()),
    T.StructField("user_id", T.LongType()),
    T.StructField("event_type", T.StringType()),
    T.StructField("amount", T.DoubleType()),
])


In [ ]:
stream_df = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "kafka:9092")
    .option("subscribe", "events.orders.v1")
    .load()
)
parsed_df = (
    stream_df
    .select(F.from_json(F.col("value").cast("string"), schema).alias("event"))
    .select("event.*")
)


In [ ]:
stream_checks = {
    "freshness": "max(event_ts) should move forward",
    "duplicates": "check duplicate event_id",
    "volume": "compare events per batch/window",
}
stream_checks
